# 06 — KNN Interpretability with LIME

## What This Notebook Covers

K-Nearest Neighbours (KNN) is the most conceptually simple of the three models in this project, yet the hardest to interpret. The reason: **KNN has no explicit learned parameters**. Unlike the Decision Tree (which learned rules) or Naïve Bayes (which learned per-class Gaussian parameters), KNN simply memorises the entire training set. When predicting for a new patient, it finds the *k* most similar training examples in the scaled feature space and takes a majority vote.

The training data *is* the model. There is nothing to open up and read.

### How KNN Makes a Prediction

For each new patient, KNN follows these steps:

1. Scale all feature values using the same `StandardScaler` fitted on the training data
2. Compute the Euclidean distance between this patient and every training patient in the scaled 11-dimensional feature space
3. Find the **k = 7** closest training patients (the "neighbours")
4. Predict by majority vote — if 5 of the 7 neighbours are *Survived*, predict *Survived* with P = 5/7 ≈ 0.71

> **Why scaling matters:** Without `StandardScaler`, features measured in large numeric ranges (e.g., Platelet Count at ~200 000) would dominate the Euclidean distance and completely drown out features in small ranges (e.g., binary Anaemia at 0 or 1). Scaling brings all features onto the same numeric footing so distances reflect genuine similarity across all clinical dimensions equally.

### Why We Need LIME

Without inspectable parameters, we cannot directly answer *"why did KNN predict this patient would die?"*. To get an explanation we need a post-hoc tool. We use **LIME** (Local Interpretable Model-agnostic Explanations), which works in five steps:

1. Start with the patient's actual feature values as a reference point
2. Generate ~1 000 slightly perturbed variants of that patient by sampling around their values
3. Run each variant through the full KNN pipeline to get predicted probabilities
4. Fit a simple **linear model** to approximate KNN's behaviour across those perturbed samples, weighted by proximity to the original patient
5. Return the coefficients of that linear model as **LIME weights** — they reveal which features pushed the prediction in each direction *for this specific patient*

### Local vs. Global Explanations

This is the most important conceptual difference from notebooks 04 and 05:

| | Decision Tree | Naïve Bayes | KNN + LIME |
|---|---|---|---|
| **Explanation scope** | Global — same rules for every patient | Global — same parameters applied to every patient | **Local** — a different explanation per patient |
| **Is the explanation exact?** | Yes — exact rule trace | Yes — exact Bayes formula | Approximate — LIME fits a linear surrogate |

The features that matter for Patient A may be entirely different from those that matter for Patient B, because KNN's decision boundary can have a different shape in different regions of the feature space.

### What We Will Explore

We select **three representative patients** from the test set and apply LIME to each:

- **Instance A** — the patient the model is most confident *Survived*
- **Instance B** — the patient the model is most confident *Deceased*
- **Instance C** — the patient the model is most uncertain about (prediction closest to 50/50)

> For plain-English descriptions of each clinical feature and its normal range, refer to the **Feature Glossary in notebook 04**.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_data, split_data
from src.models import train_knn
from src.interpret_knn import explain_instance
from IPython.display import Image

In [ ]:
X, y = load_data('../data/heart_failure_clinical_records_dataset.csv')
X_train, X_test, y_train, y_test = split_data(X, y)

pipeline = train_knn(X_train, y_train)

# Same label mapping as notebook 04 — no full glossary repeated here
FEATURE_LABELS = {
    'age':                      'Age (years)',
    'anaemia':                  'Anaemia',
    'creatinine_phosphokinase': 'CPK Enzyme Level (µg/L)',
    'diabetes':                 'Diabetes',
    'ejection_fraction':        'Ejection Fraction (%)',
    'high_blood_pressure':      'High Blood Pressure',
    'platelets':                'Platelet Count (kiloplatelets/mL)',
    'serum_creatinine':         'Serum Creatinine (mg/dL)',
    'serum_sodium':             'Serum Sodium (mEq/L)',
    'sex':                      'Sex (0=Female, 1=Male)',
    'smoking':                  'Smoking',
}

FEATURE_NAMES  = list(X.columns)   # raw names — required by the ColumnTransformer pipeline
READABLE_NAMES = [FEATURE_LABELS[f] for f in FEATURE_NAMES]
CLASS_NAMES    = ['Survived', 'Deceased']

## How to Read a LIME Explanation Chart

Before looking at any chart, here is the key to reading LIME output — it is not immediately intuitive on first encounter.

| Element | Meaning |
|---|---|
| **Each horizontal bar** | One feature's local contribution to the prediction for this specific patient |
| **Blue bar** | This feature pushed the prediction toward *Survived* |
| **Red bar** | This feature pushed the prediction toward *Deceased* |
| **Bar length** | Magnitude of influence — a longer bar means a stronger push in that direction |
| **Condition label** (e.g., `Ejection Fraction (%) <= 38.50`) | The value range LIME discretised this patient's feature into. It describes *where* the patient falls on that feature's scale |
| **Bars near zero** | That feature had little influence on this particular prediction |

> **Critical caveat — LIME weights are local.** They describe the model's behaviour in the immediate neighbourhood of one specific patient. Do not interpret them as global feature importances. The same feature can show a strong blue bar for one patient and a strong red bar for another, because KNN's decision boundary takes a different shape in different regions of the feature space.

---
## 1. Selecting Representative Instances

We deliberately choose three patients that each probe a different aspect of the model's behaviour:

- **Instance A (most confident Survived):** All or nearly all of the 7 nearest neighbours were survivors. LIME should find clear, dominant features — the patient's profile strongly resembles the surviving group. This is the "easy" case for the model.
- **Instance B (most confident Deceased):** The mirror image — the patient's profile closely resembles the deceased group. We expect strong red bars on the same features that showed strong blue bars in Instance A, but in the opposite direction.
- **Instance C (borderline):** The 7 nearest neighbours are roughly split between survivors and deceased. No single feature clearly dominates. This is the most instructive case: it reveals what happens at the decision boundary, where the model is most uncertain and a clinician's judgment would add the most value.

In [ ]:
proba = pipeline.predict_proba(X_test)
prob_survived = proba[:, 0]
prob_deceased = proba[:, 1]

idx_survived   = prob_survived.argmax()                  # most confident Survived
idx_deceased   = prob_deceased.argmax()                  # most confident Deceased
idx_borderline = (abs(prob_survived - 0.5)).argmin()     # closest to 50/50

print(f'Instance A — Confident Survived : test index {idx_survived:3d}   P(Survived) = {prob_survived[idx_survived]:.2f}')
print(f'Instance B — Confident Deceased : test index {idx_deceased:3d}   P(Deceased) = {prob_deceased[idx_deceased]:.2f}')
print(f'Instance C — Borderline         : test index {idx_borderline:3d}   P(Survived) = {prob_survived[idx_borderline]:.2f}')

In [ ]:
selected = X_test.iloc[[idx_survived, idx_deceased, idx_borderline]].copy()
selected.index   = ['A — Confident Survived', 'B — Confident Deceased', 'C — Borderline']
selected.columns = READABLE_NAMES

# Transpose so features are rows and patients are columns — easier to scan with 11 features
selected.T.round(3)

### Reading the Three Patient Profiles

Before running LIME, it is worth reading the table above directly. The clinical feature values already tell a story:

- **Ejection Fraction (%):** Compare the value across the three patients. Instance A should have the highest (heart pumping well), Instance B the lowest (heart failure territory, below 40%). Instance C likely falls somewhere in between — or has a mixed profile where some features look like A and others like B.
- **Serum Creatinine (mg/dL):** Instance B is expected to have elevated creatinine (above 1.5 mg/dL), signalling kidney stress. Instance A should be in or near the normal range (0.6–1.2 mg/dL).
- **Binary features** (Anaemia, Diabetes, etc.): These take values 0 or 1. Any differences between patients in these columns will appear as small LIME contributions, since KNN weights all features equally in distance space after scaling.

> Reading the raw feature values alongside the LIME explanations is good practice: if the model's LIME weights align with what the clinical values suggest, confidence in the explanation increases. If they conflict, it is worth investigating why.

In [ ]:
def make_label_readable(condition, label_map):
    """Replace raw column names in a LIME condition string with readable labels.
    Processes longest names first to prevent partial-match replacements."""
    for raw, readable in sorted(label_map.items(), key=lambda x: -len(x[0])):
        condition = condition.replace(raw, readable)
    return condition


def plot_lime_readable(explanation, label_map, title, path):
    """Plot a LIME explanation with readable feature labels on the axes.

    Note: explain_instance must be called with raw FEATURE_NAMES so the
    ColumnTransformer pipeline receives the column names it was trained on.
    This function post-processes the condition strings for display only.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True) if os.path.dirname(path) else None

    fw        = [(make_label_readable(c, label_map), w) for c, w in explanation.as_list()]
    fw_sorted = sorted(fw, key=lambda x: abs(x[1]))
    labels    = [x[0] for x in fw_sorted]
    weights   = [x[1] for x in fw_sorted]
    colors    = ['steelblue' if w > 0 else 'tomato' for w in weights]

    fig, ax = plt.subplots(figsize=(10, max(4, len(labels) * 0.5)))
    ax.barh(labels, weights, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('LIME weight  (positive → Survived,  negative → Deceased)')
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)

---
## 2. Instance A — Confident Survived Prediction

LIME perturbs the feature values around this patient, runs each perturbation through KNN, and fits a local linear model to approximate which features were decisive.

**What to expect:** Strong blue bars on the features that placed this patient near surviving neighbours — most likely a high Ejection Fraction and/or a low Serum Creatinine, consistent with the findings from the Decision Tree (notebook 04) and Naïve Bayes (notebook 05). A high model confidence (P close to 1.0) means most or all of the 7 neighbours were survivors, so the local linear approximation should be stable and clear.

In [ ]:
# FEATURE_NAMES (raw) is passed to explain_instance so the internal predict_fn
# can construct DataFrames that the ColumnTransformer pipeline recognises.
_, exp_a = explain_instance(
    pipeline, X_train, X_test.iloc[idx_survived], FEATURE_NAMES, CLASS_NAMES
)

plot_lime_readable(
    exp_a, FEATURE_LABELS,
    title=f'LIME — Instance A  (Predicted: Survived,  P = {prob_survived[idx_survived]:.2f})',
    path='../outputs/lime_instance_a.png'
)
Image('../outputs/lime_instance_a.png')

### Interpreting Instance A

- **Dominant blue bars** identify the features that most strongly placed this patient near the surviving group in the training set. These are the features where this patient's values fall clearly in the range associated with lower mortality risk.
- If **Ejection Fraction** appears with a strong blue bar, it means this patient's heart function is measurably better than the threshold that separated survivors from deceased in the training neighbourhood — even if the exact threshold differs from the Decision Tree's root split.
- If **Serum Creatinine** appears with a strong blue bar, the patient's kidneys are functioning well, which combined with adequate heart function paints a low-risk profile.
- **Features with near-zero bars** were not decisive for this patient in this local region. This does not mean they are globally unimportant — only that, in the neighbourhood around this particular patient, they did not shift the probability estimate significantly.

> A high confidence prediction from KNN means the neighbourhood is homogeneous — the 7 nearest training patients all point to the same outcome. LIME's linear approximation is most reliable in this setting, since the true local boundary is nearly flat.

---
## 3. Instance B — Confident Deceased Prediction

**What to expect:** The near-mirror image of Instance A — strong red bars on the features that placed this patient close to deceased training examples. We expect low Ejection Fraction and/or elevated Serum Creatinine to dominate, but the specific magnitudes and any additional contributing features may differ, because this patient lives in a different region of the feature space.

In [ ]:
_, exp_b = explain_instance(
    pipeline, X_train, X_test.iloc[idx_deceased], FEATURE_NAMES, CLASS_NAMES
)

plot_lime_readable(
    exp_b, FEATURE_LABELS,
    title=f'LIME — Instance B  (Predicted: Deceased,  P = {prob_deceased[idx_deceased]:.2f})',
    path='../outputs/lime_instance_b.png'
)
Image('../outputs/lime_instance_b.png')

### Interpreting Instance B

- **Dominant red bars** identify the features that placed this patient deep inside the deceased neighbourhood. Look at which features appear at the top of the chart and compare them to Instance A — are the same features dominant, just in the opposite direction? Cross-model consistency (same features flagged by the Decision Tree, Naïve Bayes, and now LIME) increases confidence that these features carry genuine predictive signal.
- **Features that appear in Instance B but not Instance A** (or vice versa) are interesting: they suggest the decision boundary in this region of the feature space is shaped differently from the region around Instance A. This is only possible in KNN — the tree and Naïve Bayes use the same logic everywhere.
- **Serum Creatinine elevated above ~1.5 mg/dL** is clinically serious — it indicates kidney dysfunction. Combined with a low Ejection Fraction, this is the cardiorenal syndrome pattern identified in earlier notebooks. If LIME reflects this combination with strong red bars on both features, it is a reassuring sign that the model has learned a clinically coherent pattern.

---
## 4. Instance C — Borderline Case

This is the most instructive of the three instances. A prediction near 50/50 means KNN's 7 nearest neighbours are roughly split — approximately 3–4 on one side, 3–4 on the other. The patient sits at (or very near) the decision boundary in the scaled feature space.

**What to expect:** Shorter bars overall (no single feature is overwhelmingly decisive), and a mix of blue and red bars — some features nudge toward Survived, others toward Deceased, and they partially cancel out. The overall balance of those competing forces produces the near-50/50 probability.

> **LIME reliability note:** LIME's linear approximation is least stable near a complex or curved decision boundary — exactly where Instance C lives. Small changes in the random seed or the number of perturbation samples can shift the weights noticeably. Treat the Instance C explanation as directional guidance, not a precise quantitative statement.

In [ ]:
_, exp_c = explain_instance(
    pipeline, X_train, X_test.iloc[idx_borderline], FEATURE_NAMES, CLASS_NAMES
)

plot_lime_readable(
    exp_c, FEATURE_LABELS,
    title=f'LIME — Instance C  (Borderline,  P(Survived) = {prob_survived[idx_borderline]:.2f})',
    path='../outputs/lime_instance_c.png'
)
Image('../outputs/lime_instance_c.png')

### Interpreting Instance C

- **Mixed bar colours** reflect a genuinely contested neighbourhood. The patient has some features that resemble survivors and others that resemble deceased patients — and KNN found both types among the 7 nearest neighbours.
- **Shorter bars** mean no single feature dominates. The prediction uncertainty is not a failure of the model — it accurately reflects that this patient's profile does not clearly belong to either class based on the training data.
- **Clinical implication:** This is exactly the type of patient where an ML model should flag uncertainty rather than confidently recommend a course of action. The appropriate response is escalation to clinical judgment, additional tests, or a longer observation period — not a high-confidence automated decision.
- **Compare to Instances A and B:** If the borderline patient's feature values sit between those of A and B on the table from earlier, the LIME chart should reflect this ambiguity. If unexpected features dominate the Instance C explanation, it reveals something interesting about the shape of the boundary in that region — worth noting as a finding.

---
## Key Takeaways

### What LIME Revealed

- **Cross-instance patterns:** If Ejection Fraction and Serum Creatinine dominate across all three instances (in different directions), this is strong evidence that KNN has also learned the same two features are the primary separators — consistent with the Decision Tree and Naïve Bayes. Three different algorithms finding the same signal is meaningful.
- **Instance-specific variation:** Features that appear prominently for one instance but not others demonstrate the locality of KNN explanations. The model's "reasoning" adapts to where in the feature space the patient lives.
- **The borderline case is the most valuable:** It is the instance where interpretability tools add the most practical value — precisely because the model itself is uncertain and a human needs to understand *why* before making a decision.

### LIME's Known Limitations

- **Approximation error:** LIME's linear surrogate is an approximation of the true (non-linear, non-parametric) KNN boundary. The approximation is better where the boundary is nearly flat (high-confidence predictions) and worse where it is curved or complex (borderline predictions).
- **Instability:** Running LIME twice with different random seeds may produce slightly different weights, especially for Instance C. This is a known property of the perturbation-based approach, not a bug — but it means LIME explanations should not be treated as single ground-truth answers.
- **Discretisation artefacts:** LIME discretises continuous features into ranges (e.g., `Ejection Fraction (%) <= 38.50`). The choice of bin boundaries affects which condition string appears, though the underlying signal is usually robust.

### Full Three-Model Interpretability Comparison

| Aspect | Decision Tree (04) | Naïve Bayes (05) | KNN + LIME (06) |
|---|---|---|---|
| **What you inspect** | If/else decision rules | Per-class means and standard deviations | Per-patient LIME weights (local linear approximation) |
| **Explanation scope** | Global — same rules for every patient | Global — same parameters applied to every patient | Local — a different explanation for each patient |
| **Exact or approximate?** | Exact | Exact | Approximate |
| **Features used per prediction** | 2–3 (those in the active tree path) | All 11 simultaneously | Varies by patient and local neighbourhood |
| **Handles feature interactions** | Partially — through sequential conditional splits | No — independence assumption | Implicitly — distance combines all features | 
| **Easiest to explain to a non-ML audience** | Yes — rules are intuitive | No — probability products are abstract | No — local linear surrogate requires explanation |
| **Best used for** | Auditing the full model's global logic | Understanding how each class "looks" on each feature | Debugging or explaining individual predictions |
| **Core assumption** | Greedy split optimisation | Feature independence + Gaussian distributions within each class | No distributional assumption — but LIME assumes local linearity |

### Next Steps

- **Notebook 07** brings all three models together for a side-by-side global feature importance comparison using permutation importance and SHAP. This is where we can formally ask: across three very different algorithms, which features consistently matter — and which ones only appear important in one model?